# 10 · Powerlifting：第三轮特征 + 第二层平滑实验（Colab GPU，五折）

**08 / 09 的结果**

|实验|CV RMSE|公开榜|
|---|---:|---:|
|08 五折堆叠（07 的模型 + TabM、cb_rel、lgb_rel、RealMLP，全局 NNLS）|77.013|77.207|
|+ 双胞胎|76.757|77.018|
|+ 同一运动员平滑|**76.521**|**76.731**|
|v2 + 双胞胎 + 同一运动员平滑|76.774|77.125|

- 新模型：TabM 单独 77.636，是目前最好的单模型，加入 07 融合 −0.15（5/5 折）；cb_rel 77.761（−0.06）；RealMLP −0.07 但每折 15 分钟；lgb_rel −0.04。三种堆叠方式中全局 NNLS 最好。
- 同一运动员平滑：交叉拟合 −0.24（v2 上 −0.28），公开榜 −0.29 / −0.48，是目前最有效的一步。
- 09：运动员历史 `H` −0.13（5/5 折，唯一达标）；`C`（CatBoost 的比赛 / 运动员 ID）单模型 −0.12，融合只 −0.05；`M2` 对 CatBoost、XGBoost 各 −0.06，对 LightGBM +0.07；`D`、`S` 对 XGBoost 有效（−0.08 / −0.09），融合无效。残差诊断显示剩下的分组偏差多在 2 kg 以内。

**本轮基线 `R2`** = 09 的 ALL2（F0 + P + T + M + H，41 列），模型同 09（LightGBM、XGBoost、CatBoost C2）。

|块|内容|为什么|
|---|---|---|
|`H2` 运动员生涯|这个人参加过的最大比赛规模、最多来自几个州的比赛、是否出国比赛过、换过几个级别、体重变化范围、第一次出现的年龄、同一年出现次数、参加测试赛的比例、出生年 ±1 的模糊匹配计数|`H` 有效说明「这个人是谁」重要；「去过多高级别的比赛」把比赛级别（M）和个人（H）结合起来|
|`E` 比赛的经验结构|选手平均出现次数、平均经验年数、老将（经验 ≥ 3 年）比例、新人（只出现一次）比例|经验丰富的选手多的比赛级别更高|
|`PM` 按模型挑特征|CatBoost 加 `M2` + `C`；XGBoost 加 `M2` + `S` + `D`；LightGBM 不加|09 中各模型单独有效的块不同；也增加融合多样性|

**第二层平滑实验**（只用 CPU，几分钟）：在本轮最终融合上比较 5 种做法：08 原版（精确出生年）、出生年 ±1 模糊匹配、时间衰减（离这次比赛越近的成绩权重越大）、三者合并的线性组合、小 LightGBM。全部按「性别 + 国家 + 州」分两半交叉拟合（同一运动员的所有记录在同一半）。最好的一种写进提交文件，并用于最终集成（11）。

**判读**：同 07 / 09：三模型融合平均 ΔRMSE ≤ −0.05 kg 且 ≥ 4/5 折。达标的块自动合成 `ALL3`。

**操作**：Colab 选 **T4 GPU**，从上到下运行；约 1–1.5 小时，可续跑。

## 1. 环境、GPU 与配置（与 09 相同，实验名不同）

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import importlib
import importlib.metadata
import json
import os
import shutil
import subprocess
import sys
import time
import warnings

try:
    from google.colab import drive
except ModuleNotFoundError:
    drive = None
if drive is not None:
    drive.mount('/content/drive')

PROJECT_DIR = Path(os.environ.get(
    'PL_PROJECT_DIR',
    '/content/drive/MyDrive/Colab Notebooks/performances-en-powerlifting',
)).expanduser().resolve()
if not PROJECT_DIR.is_dir():
    raise FileNotFoundError(f'找不到项目目录：{PROJECT_DIR}；请修改 PROJECT_DIR')

CATBOOST_VERSION = '1.2.10'
try:
    installed_catboost = importlib.metadata.version('catboost')
except importlib.metadata.PackageNotFoundError:
    installed_catboost = None
if installed_catboost != CATBOOST_VERSION:
    if 'catboost' in sys.modules:
        raise RuntimeError('当前运行时已经导入另一版 CatBoost；请重启运行时后从头运行')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', f'catboost=={CATBOOST_VERSION}'])
    importlib.invalidate_caches()

import catboost
import lightgbm as lgb
import numpy as np
import pandas as pd
import xgboost as xgb
from IPython.display import display
from scipy.optimize import nnls
from sklearn.model_selection import KFold, train_test_split

warnings.filterwarnings('ignore', category=UserWarning)
if catboost.__version__ != CATBOOST_VERSION:
    raise RuntimeError(f'需要 CatBoost {CATBOOST_VERSION}；实际为 {catboost.__version__}')
if shutil.which('nvidia-smi') is None:
    raise RuntimeError('未找到 nvidia-smi：请在 Colab 选择 T4 GPU 后从头运行')
gpu_probe = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                           text=True, capture_output=True, check=False)
if gpu_probe.returncode != 0 or not gpu_probe.stdout.strip():
    raise RuntimeError('NVIDIA GPU 不可用：请在 Colab 选择 T4 GPU 后从头运行')
GPU_NAME = gpu_probe.stdout.strip().splitlines()[0]

EXPERIMENT_ID = 'feature_round3_v1'
OUTPUT_DIR = PROJECT_DIR / 'artifacts' / EXPERIMENT_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SEED = 42
N_FOLDS = 5
INNER_STOP_FRACTION = 0.10
MAX_ROUNDS = 20000
MODELS = ['lgb', 'xgb', 'catboost']    # 每折内的运行顺序：快的先跑
RUN_ONLY = None                        # 例如 ['R1', 'H']；None 表示全部筛选变体
KEEP_DELTA = -0.05                     # kg：平均 ΔRMSE ≤ −0.05 且 ≥ 4/5 折更好才保留
EXPECTED_SHA256 = {'train.csv': '62319814116506fb95a51f1185dbacdb170ab055642ee4c347a30584093d0856', 'test.csv': '563c6fa29f9e535740ef1f09b25954f00fbe242c81c0e5a7343dc18aa962ba46'}
VERSIONS = {'catboost': catboost.__version__, 'xgboost': xgb.__version__, 'lightgbm': lgb.__version__,
            'numpy': np.__version__, 'pandas': pd.__version__, 'python': sys.version.split()[0]}
print('PROJECT_DIR :', PROJECT_DIR)
print('OUTPUT_DIR  :', OUTPUT_DIR)
print('GPU         :', GPU_NAME)
print('版本         :', VERSIONS)

Mounted at /content/drive
PROJECT_DIR : /content/drive/MyDrive/Colab Notebooks/performances-en-powerlifting
OUTPUT_DIR  : /content/drive/MyDrive/Colab Notebooks/performances-en-powerlifting/artifacts/feature_round3_v1
GPU         : Tesla T4, 15360 MiB
版本         : {'catboost': '1.2.10', 'xgboost': '3.4.1', 'lightgbm': '4.6.0', 'numpy': '2.1.3', 'pandas': '2.2.3', 'python': '3.13.15'}


## 2. 读取并核对数据（与 07 相同）

In [2]:
def require(condition, message):
    if not bool(condition):
        raise ValueError(message)


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def rmse(a, b):
    a, b = np.asarray(a, dtype='float64'), np.asarray(b, dtype='float64')
    return float(np.sqrt(np.mean((a - b) ** 2)))


for name, expected in EXPECTED_SHA256.items():
    require(sha256_file(PROJECT_DIR / name) == expected, f'{name} 与本地版本不同；请检查 Drive 上的数据')

TARGET = 'TotalKg'
train = pd.read_csv(PROJECT_DIR / 'train.csv')
test = pd.read_csv(PROJECT_DIR / 'test.csv')
RAW_NUM = ['Age', 'BodyweightKg', 'MeetYear']
RAW_FEATURES = [c for c in test.columns if c != 'id']
CAT = [c for c in RAW_FEATURES if c not in RAW_NUM]          # 不依赖 object / str dtype（pandas 2 与 3 都可用）
require(train.shape == (105000, 18) and test.shape == (45000, 17), '数据形状不符')
require(len(CAT) == 13 and all(pd.api.types.is_numeric_dtype(train[c]) for c in RAW_NUM)
        and not any(pd.api.types.is_numeric_dtype(train[c]) for c in CAT), '列分组不符')
require(not train[RAW_FEATURES].isna().any().any() and not test[RAW_FEATURES].isna().any().any(), '出现空值')
y = train[TARGET].to_numpy(dtype='float64')
n_train = len(train)

FOLD_IDS = np.zeros(n_train, dtype=int)
for k, (_, valid) in enumerate(KFold(N_FOLDS, shuffle=True, random_state=SEED).split(np.arange(n_train)), start=1):
    FOLD_IDS[valid] = k
print(f'train {train.shape}  test {test.shape}；每折验证行数：{np.bincount(FOLD_IDS)[1:].tolist()}')

train (105000, 18)  test (45000, 17)；每折验证行数：[21000, 21000, 21000, 21000, 21000]


## 3. 第一轮特征（与 07 相同）

In [3]:
both = pd.concat([train[RAW_FEATURES], test[RAW_FEATURES]], ignore_index=True)

# F0：v2 的体重级别特征
weight_class = pd.to_numeric(both['WeightClassKg'].str.replace('+', '', regex=False), errors='coerce')
F0_EXTRA = pd.DataFrame({
    'WeightClass_num': weight_class,
    'Is_SHW': both['WeightClassKg'].str.endswith('+').astype('float64'),
    'BW_minus_Class': both['BodyweightKg'] - weight_class,
})

# P：记录精度（树按阈值切数值，学不到「是不是 .5」「是不是整数」）
age2 = (both['Age'] * 2).round()
require(np.allclose(both['Age'] * 2, age2), 'Age 不是 0.5 的整数倍')
bw = both['BodyweightKg']
is_int = np.isclose(bw, bw.round())
is_1dec = np.isclose(bw * 10, (bw * 10).round())
is_half = is_1dec & ~is_int & np.isclose((bw * 10).round() % 10, 5)
pounds = bw / 0.45359237
P = pd.DataFrame({
    'age_half': (age2 % 2).astype('float64'),
    'bw_precision': np.select([is_int, is_half, is_1dec], [0.0, 1.0, 2.0], default=3.0),
    'bw_lb_int': (np.abs(pounds - pounds.round()) < 0.015).astype('float64'),
})

# T：跨地参赛（居住地未知时为 NaN）
abroad = (both['Country'] != both['MeetCountry']).astype('float64').where(both['Country'] != 'Unknown')
state_known = ((both['State'] != 'Unknown') & (both['MeetState'] != 'Unknown')
               & (both['Country'] == both['MeetCountry']))
other_state = (both['State'] != both['MeetState']).astype('float64').where(state_known)
T = pd.DataFrame({'abroad': abroad, 'other_state': other_state})

# M：比赛背景；「比赛」≈ 联盟 × 年份 × 比赛国 × 比赛州（train+test 无标签统计）
meet = (both['Federation'] + '|' + both['MeetYear'].astype(str) + '|'
        + both['MeetCountry'] + '|' + both['MeetState'])


def by_meet(values):
    return values.groupby(meet).transform('mean')


M = pd.DataFrame({
    'meet_n': meet.map(meet.value_counts()).astype('float64'),
    'meet_sexeq_n': both.groupby([meet, both['Sex'], both['Equipment']])['Age'].transform('size').astype('float64'),
    'fed_year_n': both.groupby(['Federation', 'MeetYear'])['Age'].transform('size').astype('float64'),
    'meet_share_other_state': by_meet(other_state),
    'meet_share_abroad': by_meet(abroad),
    'meet_share_age_half': by_meet(P['age_half']),
    'meet_share_equipped': by_meet((both['Equipment'] != 'Raw').astype('float64')),
    'meet_share_male': by_meet((both['Sex'] == 'M').astype('float64')),
    'meet_mean_age': by_meet(both['Age']),
})

BLOCKS = {'P': P, 'T': T, 'M': M}
VARIANTS = {'F0': [], 'P': ['P'], 'T': ['T'], 'M': ['M'], 'PT': ['P', 'T'], 'ALL': ['P', 'T', 'M']}
for name, frame in {'F0': F0_EXTRA, **BLOCKS}.items():
    require(len(frame) == len(both) and frame.index.equals(both.index), f'{name} 行数或索引不符')
    require(not np.isinf(frame.to_numpy(dtype='float64')).any(), f'{name} 含无穷大')

_FRAMES = {}


def frames_for(variant, model):
    """(train, test) 特征表。CatBoost 的类别为字符串；LGB / XGB 为 category（词表来自 train+test，无标签）。"""
    kind = 'str' if model == 'catboost' else 'category'
    if (variant, kind) not in _FRAMES:
        frame = pd.concat([both[RAW_NUM], F0_EXTRA] + [BLOCKS[b] for b in VARIANTS[variant]] + [both[CAT]], axis=1)
        require(frame.columns.is_unique, f'{variant} 列名重复')
        frame[CAT] = frame[CAT].astype(str).astype(object) if kind == 'str' else frame[CAT].astype('category')
        _FRAMES[(variant, kind)] = (frame.iloc[:n_train].reset_index(drop=True),
                                    frame.iloc[n_train:].reset_index(drop=True))
    return _FRAMES[(variant, kind)]


print({v: frames_for(v, 'lgb')[0].shape[1] for v in VARIANTS})
summary_new = pd.concat([P, T, M], axis=1)
display(pd.DataFrame({'train_mean': summary_new.iloc[:n_train].mean(), 'test_mean': summary_new.iloc[n_train:].mean(),
                      'nan_share': summary_new.isna().mean()}).round(4))

{'F0': 19, 'P': 22, 'T': 21, 'M': 28, 'PT': 24, 'ALL': 33}


,train_mean,test_mean,nan_share
age_half,0.4817,0.4782,0.0000
bw_precision,1.9668,1.9609,0.0000
bw_lb_int,0.0500,0.0503,0.0000
abroad,0.0824,0.0822,0.0184
other_state,0.2670,0.2724,0.6491
meet_n,63.6623,63.9177,0.0000
meet_sexeq_n,33.0578,33.1368,0.0000
fed_year_n,653.6460,656.8197,0.0000
meet_share_other_state,0.2661,0.2673,0.6219
meet_share_abroad,0.0833,0.0837,0.0004


## 4. 第二轮特征（与 09 相同）

In [4]:
_FRAMES.clear()                     # 07 的变体表不再需要，释放内存
CORE9 = ['Age', 'BodyweightKg', 'MeetYear', 'Sex', 'Federation', 'MeetCountry', 'MeetState', 'Country', 'State']
sex, equipment = both['Sex'], both['Equipment']
state = both['State'].where(both['State'] != 'Unknown')
country = both['Country'].where(both['Country'] != 'Unknown')
national = both['Federation'] + '|' + both['MeetYear'].astype(str) + '|' + both['MeetCountry']


def counts(key):
    return key.map(key.value_counts()).astype('float64')


def n_distinct(values, key):
    return values.groupby(key).transform('nunique').astype('float64')


# M2：比赛背景加强（meet 与 by_meet 来自 07 的特征单元）
M2 = pd.DataFrame({
    'meet_n_states': n_distinct(state, meet),
    'meet_n_countries': n_distinct(country, meet),
    'meet_n_wclass': n_distinct(both['WeightClassKg'], meet),
    'meet_share_tested': by_meet((both['Tested'] == 'Yes').astype('float64')),
    'meet_share_raw': by_meet((equipment == 'Raw').astype('float64')),
    'meet_share_multi': by_meet((equipment == 'Multi-ply').astype('float64')),
    'meet_share_bw2dec': by_meet((P['bw_precision'] == 3).astype('float64')),
    'meet_share_junior': by_meet((both['Age'] < 24).astype('float64')),
    'meet_share_master': by_meet((both['Age'] >= 40).astype('float64')),
    'meet_age_std': both['Age'].groupby(meet).transform('std'),
    'meet_sex_bw_mean': both['BodyweightKg'].groupby([meet, sex]).transform('mean'),
    'nat_n': counts(national),
    'nat_n_meetstates': n_distinct(both['MeetState'], national),
    'nat_n_states': n_distinct(state, national),
})

# D：组别背景
division = meet + '|' + sex + '|' + equipment + '|' + both['WeightClassKg']
age_division = meet + '|' + sex + '|' + equipment + '|' + both['AgeClass']
exact_key = pd.Series(pd.util.hash_pandas_object(both[RAW_FEATURES], index=False).to_numpy())
core_key = pd.Series(pd.util.hash_pandas_object(both[CORE9], index=False).to_numpy())
D = pd.DataFrame({
    'div_n': counts(division),
    'div_bw_rank': both['BodyweightKg'].groupby(division).rank(pct=True),
    'div_age_rank': both['Age'].groupby(division).rank(pct=True),
    'agediv_n': counts(age_division),
    'dup_exact_n': counts(exact_key) - 1,
    'dup_core_n': counts(core_key) - 1,
})

# H：近似同一运动员的历史（出生年 = floor(MeetYear − Age)：x.5 年龄时恰为出生年，整数年龄可能差 1）
valid = (both['Country'] != 'Unknown').to_numpy()
birth = np.floor(both['MeetYear'] - both['Age'] + 1e-9).astype(int).astype(str)
athlete = sex + '|' + both['Country'] + '|' + both['State'] + '|' + both['Federation'] + '|' + birth
athlete0 = sex + '|' + both['Country'] + '|' + both['State'] + '|' + birth
year = both['MeetYear'].astype('float64')
H = pd.DataFrame({
    'ath_n': counts(athlete),
    'ath_n_years': n_distinct(year, athlete),
    'ath_years_since_first': year - year.groupby(athlete).transform('min'),
    'ath_years_to_last': year.groupby(athlete).transform('max') - year,
    'ath_n_equipment': n_distinct(equipment, athlete),
    'ath_bw_delta': both['BodyweightKg'] - both['BodyweightKg'].groupby(athlete).transform('mean'),
    'ath0_n': counts(athlete0),
    'ath0_n_feds': n_distinct(both['Federation'], athlete0),
})
H.loc[~valid] = np.nan


# S：DOTS 系数（TotalKg = DOTS 分 × dots_scale）
def dots_scale(sex_values, bw_values):
    s = np.asarray(sex_values).astype(str)
    bw = np.asarray(bw_values, dtype='float64')
    m, f = np.clip(bw, 40, 210), np.clip(bw, 40, 150)
    poly_m = -1.0930e-6 * m**4 + 7.391293e-4 * m**3 - 0.1918759221 * m**2 + 24.0900756 * m - 307.75076
    poly_f = -1.0706e-6 * f**4 + 5.158568e-4 * f**3 - 0.1126655495 * f**2 + 13.6175032 * f - 57.96288
    return np.where(s == 'M', poly_m, np.where(s == 'F', poly_f, 0.5 * (poly_m + poly_f))) / 500.0


S = pd.DataFrame({'dots_scale': dots_scale(sex, both['BodyweightKg'])})

# C：只给 CatBoost 的高基数 ID（有序目标统计编码）
C = pd.DataFrame({'meet_id': meet, 'athlete_id': (athlete + '|' + equipment).where(valid, 'unknown')})

BLOCKS2 = {'M2': M2, 'D': D, 'H': H, 'S': S}
CB_ONLY = {'C': C}
for name, frame in {**BLOCKS2, **CB_ONLY}.items():
    require(len(frame) == len(both) and frame.index.equals(both.index), f'{name} 行数或索引不符')
for name, frame in BLOCKS2.items():
    require(not np.isinf(frame.to_numpy(dtype='float64')).any(), f'{name} 含无穷大')
require(S['dots_scale'].min() > 0.3, 'DOTS 系数异常')

R1_BLOCKS = ['P', 'T', 'M']
VARIANT_BLOCKS = {'R1': [], 'M2': ['M2'], 'D': ['D'], 'H': ['H'], 'S': ['S'], 'C': ['C']}
MODELS_FOR = {'C': ['catboost']}      # C 只改变 CatBoost；融合时 LightGBM / XGBoost 用 R1
SCREEN = [v for v in VARIANT_BLOCKS if RUN_ONLY is None or v in RUN_ONLY or v == 'R1']
_FRAMES2 = {}


def models_for(variant):
    return MODELS_FOR.get(variant, MODELS)


def frames2(variant, model):
    """(train, test, 类别列)。R1 = F0 + P + T + M；C 块只加给 CatBoost。"""
    blocks = VARIANT_BLOCKS[variant]
    kind = 'str' if model == 'catboost' else 'category'
    key = (tuple(blocks), kind)
    if key not in _FRAMES2:
        nums = ([both[RAW_NUM], F0_EXTRA] + [BLOCKS[b] for b in R1_BLOCKS]
                + [BLOCKS2[b] for b in blocks if b in BLOCKS2])
        cats = [both[CAT]] + ([CB_ONLY[b] for b in blocks if b in CB_ONLY] if kind == 'str' else [])
        frame = pd.concat(nums + cats, axis=1)
        require(frame.columns.is_unique, f'{variant} 列名重复')
        cat_cols = [c for part in cats for c in part.columns]
        if kind == 'str':
            frame[cat_cols] = frame[cat_cols].astype(str).astype(object)
        else:
            frame[cat_cols] = frame[cat_cols].astype('category')
        _FRAMES2[key] = (frame.iloc[:n_train].reset_index(drop=True),
                         frame.iloc[n_train:].reset_index(drop=True), cat_cols)
    return _FRAMES2[key]


new_cols = pd.concat([M2, D, H, S], axis=1)
display(pd.DataFrame({'train_mean': new_cols.iloc[:n_train].mean(), 'test_mean': new_cols.iloc[n_train:].mean(),
                      'nan_share': new_cols.isna().mean()}).round(4))
print('C 块类别数：', {c: int(C[c].nunique()) for c in C.columns})
print('各变体列数（LightGBM / XGBoost）：', {v: frames2(v, 'lgb')[0].shape[1] for v in VARIANT_BLOCKS if v != 'C'},
      '｜CatBoost C：', frames2('C', 'catboost')[0].shape[1])

,train_mean,test_mean,nan_share
meet_n_states,3.2039,3.2371,0.0000
meet_n_countries,2.0094,2.0170,0.0000
meet_n_wclass,10.7194,10.7360,0.0000
meet_share_tested,0.7627,0.7604,0.0000
meet_share_raw,0.6208,0.6212,0.0000
meet_share_multi,0.0234,0.0240,0.0000
meet_share_bw2dec,0.2949,0.2969,0.0000
meet_share_junior,0.4549,0.4538,0.0000
meet_share_master,0.1703,0.1714,0.0000
meet_age_std,10.0628,10.0618,0.0104


C 块类别数： {'meet_id': 10546, 'athlete_id': 46893}
各变体列数（LightGBM / XGBoost）： {'R1': 33, 'M2': 47, 'D': 39, 'H': 41, 'S': 34} ｜CatBoost C： 35


## 5. 第三轮特征块

In [5]:
_FRAMES2.clear()                    # 09 的变体表不再需要，释放内存
birth_num = np.floor(both['MeetYear'] - both['Age'] + 1e-9).astype(int)


def fuzzy_count(prefix_key):
    """同前缀、出生年为 b−1、b、b+1 的行数（整数年龄推算的出生年可能差 1）。"""
    exact = prefix_key + '|' + birth_num.astype(str)
    table = exact.value_counts()
    total = exact.map(table).astype('float64')
    for d in (-1, 1):
        total = total + (prefix_key + '|' + (birth_num + d).astype(str)).map(table).fillna(0)
    return total


# H2：运动员生涯（athlete、valid 来自 09 的特征单元；M、T 来自 07；M2 来自 09）
bw = both['BodyweightKg']
H2 = pd.DataFrame({
    'ath_max_meet_states': M2['meet_n_states'].groupby(athlete).transform('max'),
    'ath_mean_meet_states': M2['meet_n_states'].groupby(athlete).transform('mean'),
    'ath_max_meet_n': M['meet_n'].groupby(athlete).transform('max'),
    'ath_max_nat_states': M2['nat_n_states'].groupby(athlete).transform('max'),
    'ath_ever_abroad': T['abroad'].groupby(athlete).transform('max'),
    'ath_n_wclass': n_distinct(both['WeightClassKg'], athlete),
    'ath_bw_range': bw.groupby(athlete).transform('max') - bw.groupby(athlete).transform('min'),
    'ath_first_age': both['Age'].groupby(athlete).transform('min'),
    'ath_n_same_year': both.groupby([athlete, both['MeetYear']])['Age'].transform('size').astype('float64'),
    'ath_share_tested': (both['Tested'] == 'Yes').astype('float64').groupby(athlete).transform('mean'),
    'ath_n_fuzzy': fuzzy_count(sex + '|' + both['Country'] + '|' + both['State'] + '|' + both['Federation']),
    'ath0_n_fuzzy': fuzzy_count(sex + '|' + both['Country'] + '|' + both['State']),
})
H2.loc[~valid] = np.nan

# E：比赛的经验结构
experience = H['ath_years_since_first']
E = pd.DataFrame({
    'meet_mean_ath_n': H['ath_n'].groupby(meet).transform('mean'),
    'meet_mean_ath_exp': experience.groupby(meet).transform('mean'),
    'meet_share_veterans': (experience >= 3).astype('float64').where(experience.notna()).groupby(meet).transform('mean'),
    'meet_share_newcomers': (H['ath_n'] == 1).astype('float64').where(H['ath_n'].notna()).groupby(meet).transform('mean'),
})

ROUND3 = {'H2': H2, 'E': E}
for name, frame in ROUND3.items():
    require(len(frame) == len(both) and frame.index.equals(both.index), f'{name} 行数或索引不符')
    require(not np.isinf(frame.to_numpy(dtype='float64')).any(), f'{name} 含无穷大')
NUMERIC_BLOCKS = {**BLOCKS, **BLOCKS2, **ROUND3}          # P T M（07）、M2 D H S（09）、H2 E（本轮）
R2_BLOCKS = ['P', 'T', 'M', 'H']                          # 09 的 ALL2
EXTRAS = {'lgb': [], 'xgb': ['M2', 'S', 'D'], 'catboost': ['M2', 'C']}   # PM：09 中各模型单独有效的块
VARIANTS3 = {'R2': [], 'H2': ['H2'], 'E': ['E'], 'PM': ['PM']}
SCREEN3 = [v for v in VARIANTS3 if RUN_ONLY is None or v in RUN_ONLY or v == 'R2']
_FRAMES3 = {}


def blocks_for(variant, model):
    out = list(R2_BLOCKS)
    for b in VARIANTS3[variant]:
        out += EXTRAS[model] if b == 'PM' else [b]
    return out


def runs(model, variant):
    """该模型在这个变体上的特征与 R2 不同才需要训练（例如 PM 对 LightGBM 没有变化）。"""
    return variant == 'R2' or blocks_for(variant, model) != blocks_for('R2', model)


def frames3(variant, model):
    """(train, test, 类别列)。"""
    blocks = blocks_for(variant, model)
    kind = 'str' if model == 'catboost' else 'category'
    key = (tuple(blocks), kind)
    if key not in _FRAMES3:
        nums = [both[RAW_NUM], F0_EXTRA] + [NUMERIC_BLOCKS[b] for b in blocks if b in NUMERIC_BLOCKS]
        cats = [both[CAT]] + ([CB_ONLY[b] for b in blocks if b in CB_ONLY] if kind == 'str' else [])
        frame = pd.concat(nums + cats, axis=1)
        require(frame.columns.is_unique, f'{variant} {model} 列名重复')
        cat_cols = [c for part in cats for c in part.columns]
        if kind == 'str':
            frame[cat_cols] = frame[cat_cols].astype(str).astype(object)
        else:
            frame[cat_cols] = frame[cat_cols].astype('category')
        _FRAMES3[key] = (frame.iloc[:n_train].reset_index(drop=True),
                         frame.iloc[n_train:].reset_index(drop=True), cat_cols)
    return _FRAMES3[key]


new_cols = pd.concat([H2, E], axis=1)
display(pd.DataFrame({'train_mean': new_cols.iloc[:n_train].mean(), 'test_mean': new_cols.iloc[n_train:].mean(),
                      'nan_share': new_cols.isna().mean()}).round(4))
print({f'{v}/{m}': frames3(v, m)[0].shape[1] for v in VARIANTS3 for m in MODELS if runs(m, v)})

,train_mean,test_mean,nan_share
ath_max_meet_states,7.2553,7.2965,0.0184
ath_mean_meet_states,3.2682,3.2702,0.0184
ath_max_meet_n,112.4144,112.6788,0.0184
ath_max_nat_states,17.7174,17.7553,0.0184
ath_ever_abroad,0.1208,0.1206,0.0184
ath_n_wclass,6.3779,6.4201,0.0184
ath_bw_range,48.6372,49.1032,0.0184
ath_first_age,24.1705,24.0925,0.0184
ath_n_same_year,4.9418,4.9522,0.0184
ath_share_tested,0.7628,0.7612,0.0184


{'R2/lgb': 41, 'R2/xgb': 41, 'R2/catboost': 41, 'H2/lgb': 53, 'H2/xgb': 53, 'H2/catboost': 53, 'E/lgb': 45, 'E/xgb': 45, 'E/catboost': 45, 'PM/xgb': 62, 'PM/catboost': 57}


## 6. 模型与预检（参数与 09 相同：LightGBM、XGBoost 用 v2 参数，CatBoost 用 C2）

In [6]:
LGB_PARAMS = dict(learning_rate=0.05, num_leaves=63, min_child_samples=40, subsample=0.8, subsample_freq=1,
                  colsample_bytree=0.8, cat_smooth=20, verbose=-1)
XGB_PARAMS = dict(learning_rate=0.03, max_depth=8, min_child_weight=10.0, subsample=0.8, colsample_bytree=0.7,
                  reg_lambda=1.0, tree_method='hist', device='cuda', enable_categorical=True,
                  max_cat_to_onehot=1, eval_metric='rmse')
CB_PARAMS = dict(learning_rate=0.05, depth=8, l2_leaf_reg=10.0, bootstrap_type='Bernoulli', subsample=0.8,
                 border_count=128)                                   # 07 的 C2
PATIENCE = {'lgb': 200, 'xgb': 200, 'catboost': 300}


def fit_predict(model, Xa, ya, Xs, ys, Xv, Xt, cat_cols, rounds=MAX_ROUNDS):
    """在 (Xa, ya) 上训练、用 (Xs, ys) 早停；返回 (外层验证预测, 测试预测, 最佳轮数)。"""
    if model == 'lgb':
        m = lgb.LGBMRegressor(n_estimators=rounds, random_state=SEED, n_jobs=-1, **LGB_PARAMS)
        m.fit(Xa, ya, eval_set=[(Xs, ys)], callbacks=[lgb.early_stopping(PATIENCE['lgb'], verbose=False)])
        best = int(m.best_iteration_ or rounds)
        return m.predict(Xv, num_iteration=best), m.predict(Xt, num_iteration=best), best
    if model == 'xgb':
        m = xgb.XGBRegressor(n_estimators=rounds, random_state=SEED, early_stopping_rounds=PATIENCE['xgb'],
                             **XGB_PARAMS)
        m.fit(Xa, ya, eval_set=[(Xs, ys)], verbose=False)
        best = int(m.best_iteration) + 1
        return m.predict(Xv, iteration_range=(0, best)), m.predict(Xt, iteration_range=(0, best)), best
    if model == 'catboost':
        m = catboost.CatBoostRegressor(iterations=rounds, loss_function='RMSE', eval_metric='RMSE',
                                       od_type='Iter', od_wait=PATIENCE['catboost'], random_seed=SEED,
                                       verbose=0, task_type='GPU', devices='0', allow_writing_files=False,
                                       **CB_PARAMS)
        m.fit(catboost.Pool(Xa, ya, cat_features=cat_cols), eval_set=catboost.Pool(Xs, ys, cat_features=cat_cols),
              use_best_model=True)
        best = int(m.get_best_iteration()) + 1
        return m.predict(Xv), m.predict(Xt), best
    raise ValueError(model)



rng = np.random.default_rng(SEED)
probe = rng.choice(n_train, 3000, replace=False)
pa, ps, pv = probe[:2000], probe[2000:2500], probe[2500:]
failed = {}
for variant in SCREEN3:
    for model in MODELS:
        if not runs(model, variant):
            continue
        Xtr, Xte, cat_cols = frames3(variant, model)
        try:
            p_val, p_test, _ = fit_predict(model, Xtr.iloc[pa], y[pa], Xtr.iloc[ps], y[ps], Xtr.iloc[pv],
                                           Xte.iloc[:500], cat_cols, rounds=30)
            require(np.isfinite(p_val).all() and np.isfinite(p_test).all(), '预测含非有限值')
            require(len(p_val) == len(pv) and len(p_test) == 500, '预测长度不符')
            print(f'预检通过：{model:8s} {variant:3s}（{Xtr.shape[1]} 列）RMSE@30轮 {rmse(y[pv], p_val):.1f}')
        except Exception as exc:
            failed[(model, variant)] = repr(exc)
            print(f'预检失败，跳过：{model} {variant} → {exc!r}')
require(not any(v == 'R2' for (_, v) in failed), '基线 R2 预检失败，请检查环境')

预检通过：lgb      R2 （41 列）RMSE@30轮 91.5
预检通过：xgb      R2 （41 列）RMSE@30轮 116.0
预检通过：catboost R2 （41 列）RMSE@30轮 98.5
预检通过：lgb      H2 （53 列）RMSE@30轮 91.4
预检通过：xgb      H2 （53 列）RMSE@30轮 117.4
预检通过：catboost H2 （53 列）RMSE@30轮 97.4
预检通过：lgb      E  （45 列）RMSE@30轮 90.9
预检通过：xgb      E  （45 列）RMSE@30轮 116.6
预检通过：catboost E  （45 列）RMSE@30轮 97.6
预检通过：xgb      PM （62 列）RMSE@30轮 116.9
预检通过：catboost PM （57 列）RMSE@30轮 97.8


## 7. 筛选：R2 与每个新块单独加入（五折，可续跑）

In [ ]:
CONFIG = {'purpose': 'feature round 3 on top of 09 ALL2; inner 10% early stop, score early-stopped model on outer fold',
          'seed': SEED, 'n_folds': N_FOLDS, 'inner_stop_fraction': INNER_STOP_FRACTION,
          'r2_blocks': R2_BLOCKS, 'extras': EXTRAS, 'screen': VARIANTS3, 'data_sha256': EXPECTED_SHA256}
config_path = OUTPUT_DIR / 'config.json'
if config_path.is_file():
    require(json.loads(config_path.read_text(encoding='utf-8')) == CONFIG,
            f'{OUTPUT_DIR.name} 已有不同配置的结果；改动配置时请改 EXPERIMENT_ID')
else:
    config_path.write_text(json.dumps(CONFIG, ensure_ascii=False, indent=1), encoding='utf-8')


def spec_of(model, variant):
    Xtr, _, cat_cols = frames3(variant, model)
    params = {'lgb': LGB_PARAMS, 'xgb': XGB_PARAMS, 'catboost': CB_PARAMS}[model]
    return {'model': model, 'variant': variant, 'blocks': blocks_for(variant, model),
            'columns': list(Xtr.columns), 'cat_cols': list(cat_cols), 'params': dict(params),
            'patience': PATIENCE[model]}


def run_one(model, variant, fold):
    stem = f'{model}__{variant}__fold{fold}'
    json_path, npz_path = OUTPUT_DIR / f'{stem}.json', OUTPUT_DIR / f'{stem}.npz'
    spec = spec_of(model, variant)
    if json_path.is_file() and npz_path.is_file():
        saved = json.loads(json_path.read_text(encoding='utf-8'))
        require(saved['spec'] == spec, f'{stem} 的定义已改变；请改 EXPERIMENT_ID 或变体名')
        return saved, 'cache'
    Xtr, Xte, cat_cols = frames3(variant, model)
    fit_idx = np.flatnonzero(FOLD_IDS != fold)
    valid_idx = np.flatnonzero(FOLD_IDS == fold)
    inner_fit, inner_stop = train_test_split(fit_idx, test_size=INNER_STOP_FRACTION, random_state=SEED)  # = 07
    started = time.perf_counter()
    p_val, p_test, best = fit_predict(model, Xtr.iloc[inner_fit], y[inner_fit], Xtr.iloc[inner_stop],
                                      y[inner_stop], Xtr.iloc[valid_idx], Xte, cat_cols)
    require(np.isfinite(p_val).all() and np.isfinite(p_test).all(), f'{stem} 预测含非有限值')
    np.savez(npz_path, valid_idx=valid_idx, val=np.asarray(p_val, dtype='float64'),
             test=np.asarray(p_test, dtype='float64'))
    result = {'model': model, 'variant': variant, 'fold': fold, 'spec': spec, 'n_features': int(Xtr.shape[1]),
              'best_iteration': best, 'hit_limit': bool(best >= MAX_ROUNDS - PATIENCE[model]),
              'rmse': rmse(y[valid_idx], p_val), 'seconds': time.perf_counter() - started,
              'gpu': GPU_NAME, 'versions': VERSIONS, 'finished_at_utc': datetime.now(timezone.utc).isoformat()}
    json_path.write_text(json.dumps(result, ensure_ascii=False, indent=1), encoding='utf-8')
    return result, f'{result["seconds"]:.0f}s'


def run_variants(variants):
    for fold in range(1, N_FOLDS + 1):
        fold_started = time.perf_counter()
        for model in MODELS:
            for variant in variants:
                if not runs(model, variant) or (model, variant) in failed:
                    continue
                result, how = run_one(model, variant, fold)
                print(f'fold {fold}  {model:8s} {variant:4s}  RMSE {result["rmse"]:.3f}  '
                      f'最佳轮数 {result["best_iteration"]:5d}  ({how})')
        print(f'—— fold {fold} 完成，本次运行用时 {(time.perf_counter() - fold_started) / 60:.1f} 分钟')


run_variants(SCREEN3)

fold 1  lgb      R2    RMSE 77.833  最佳轮数   497  (18s)
fold 1  lgb      H2    RMSE 77.778  最佳轮数   448  (25s)
fold 1  lgb      E     RMSE 77.749  最佳轮数   464  (17s)
fold 1  xgb      R2    RMSE 78.399  最佳轮数   539  (8s)
fold 1  xgb      H2    RMSE 78.353  最佳轮数   631  (10s)
fold 1  xgb      E     RMSE 78.290  最佳轮数   573  (9s)
fold 1  xgb      PM    RMSE 78.512  最佳轮数   581  (10s)
fold 1  catboost R2    RMSE 77.786  最佳轮数  3309  (119s)
fold 1  catboost H2    RMSE 77.730  最佳轮数  3204  (114s)
fold 1  catboost E     RMSE 77.688  最佳轮数  3718  (132s)
fold 1  catboost PM    RMSE 77.488  最佳轮数  3537  (139s)
—— fold 1 完成，本次运行用时 10.0 分钟
fold 2  lgb      R2    RMSE 78.051  最佳轮数   373  (13s)
fold 2  lgb      H2    RMSE 78.074  最佳轮数   377  (15s)
fold 2  lgb      E     RMSE 77.929  最佳轮数   333  (13s)
fold 2  xgb      R2    RMSE 78.879  最佳轮数   435  (7s)
fold 2  xgb      H2    RMSE 78.972  最佳轮数   419  (8s)
fold 2  xgb      E     RMSE 78.955  最佳轮数   457  (8s)
fold 2  xgb      PM    RMSE 79.087  最佳轮数   434  (9s)
fo

## 8. 与 R2 配对比较（单模型 + 三模型融合），自动合成 ALL3

In [ ]:
def load_oof(model, variant):
    """五折都完成时返回 (OOF, 五折平均测试预测)，否则 None。"""
    oof, test_pred = np.full(n_train, np.nan), np.zeros(len(test))
    for fold in range(1, N_FOLDS + 1):
        path = OUTPUT_DIR / f'{model}__{variant}__fold{fold}.npz'
        if not path.is_file():
            return None
        saved = np.load(path)
        oof[saved['valid_idx']] = saved['val']
        test_pred += saved['test'] / N_FOLDS
    require(not np.isnan(oof).any(), f'{model} {variant} 的 OOF 不完整')
    return oof, test_pred


def fold_rmse(pred):
    return pd.Series({k: rmse(y[FOLD_IDS == k], pred[FOLD_IDS == k]) for k in range(1, N_FOLDS + 1)})


def paired_table(preds, base='R2'):
    require(base in preds, f'缺少基线 {base}')
    base_folds = fold_rmse(preds[base])
    rows = []
    for name, pred in preds.items():
        delta = fold_rmse(pred) - base_folds
        rows.append({'variant': name, 'oof_rmse': rmse(y, pred), 'mean_delta': delta.mean(),
                     'folds_better': int((delta < 0).sum()), **{f'd_fold{k}': v for k, v in delta.items()}})
    table = pd.DataFrame(rows).set_index('variant')
    table['keep'] = (table['mean_delta'] <= KEEP_DELTA) & (table['folds_better'] >= 4)
    table.loc[base, 'keep'] = False
    return table.sort_values('oof_rmse')


def blend_variant(variant):
    """三模型 NNLS 融合；某模型在该变体上特征不变时用它的 R2 结果。"""
    loaded = []
    for model in MODELS:
        item = load_oof(model, variant if runs(model, variant) else 'R2')
        if item is None:
            return None
        loaded.append(item)
    P_oof = np.column_stack([o for o, _ in loaded])
    P_test = np.column_stack([t for _, t in loaded])
    w, _ = nnls(P_oof, y)
    return {'oof': P_oof @ w, 'test': P_test @ w, 'weights': [round(float(v), 3) for v in w]}


results = pd.DataFrame([json.loads(p.read_text(encoding='utf-8')) for p in sorted(OUTPUT_DIR.glob('*__fold*.json'))])
for model in MODELS:
    preds = {}
    for variant in VARIANTS3:
        item = load_oof(model, variant) if runs(model, variant) else None
        if item is not None:
            preds[variant] = item[0]
    if 'R2' in preds:
        table = paired_table(preds)
        part = results[results['model'] == model].groupby('variant')
        table['mean_best_iter'] = part['best_iteration'].mean()
        table['minutes'] = part['seconds'].sum() / 60
        print(f'\n=== {model}')
        display(table.round(3))
        table.to_csv(OUTPUT_DIR / f'summary_single_{model}.csv')

BLENDS = {}
for variant in VARIANTS3:
    item = blend_variant(variant)
    if item is not None:
        BLENDS[variant] = item
require('R2' in BLENDS, '三模型的 R2 还没有全部完成')
blend_table = paired_table({v: b['oof'] for v, b in BLENDS.items()})
blend_table['weights_lgb_xgb_cb'] = [str(BLENDS[v]['weights']) for v in blend_table.index]
print('\n=== 三模型 NNLS 融合（主要判读依据）')
display(blend_table.round(3))
blend_table.to_csv(OUTPUT_DIR / 'summary_blend_screen.csv')

KEPT = [v for v in ['H2', 'E', 'PM'] if v in blend_table.index and bool(blend_table.loc[v, 'keep'])]
VARIANTS3['ALL3'] = KEPT
print('达标的新块：', KEPT or '无', '→ ALL3 = R2 +', KEPT)


=== lgb


,oof_rmse,mean_delta,folds_better,d_fold1,d_fold2,d_fold3,d_fold4,d_fold5,keep,mean_best_iter,minutes
variant,,,,,,,,,,,
E,77.765,-0.108,5,-0.084,-0.122,-0.138,-0.062,-0.133,True,405.2,1.289
H2,77.827,-0.046,3,-0.055,0.023,-0.298,0.183,-0.085,False,411.6,1.524
R2,77.873,0.000,0,0.000,0.000,0.000,0.000,0.000,False,448.8,1.290



=== xgb


,oof_rmse,mean_delta,folds_better,d_fold1,d_fold2,d_fold3,d_fold4,d_fold5,keep,mean_best_iter,minutes
variant,,,,,,,,,,,
E,78.500,-0.064,4,-0.110,0.076,-0.108,-0.049,-0.128,True,494.8,0.684
R2,78.563,0.000,0,0.000,0.000,0.000,0.000,0.000,False,499.0,0.637
H2,78.588,0.025,2,-0.047,0.093,0.033,0.074,-0.029,False,521.8,0.741
PM,78.657,0.094,0,0.112,0.207,0.072,0.071,0.007,False,468.6,0.752



=== catboost


,oof_rmse,mean_delta,folds_better,d_fold1,d_fold2,d_fold3,d_fold4,d_fold5,keep,mean_best_iter,minutes
variant,,,,,,,,,,,
PM,77.557,-0.222,5,-0.298,-0.282,-0.128,-0.263,-0.137,True,3111.8,10.281
H2,77.708,-0.070,4,-0.056,-0.133,-0.113,-0.058,0.008,True,2799.0,8.367
E,77.734,-0.045,3,-0.098,-0.064,-0.115,0.021,0.032,False,3135.8,9.320
R2,77.779,0.000,0,0.000,0.000,0.000,0.000,0.000,False,3003.2,8.976



=== 三模型 NNLS 融合（主要判读依据）


,oof_rmse,mean_delta,folds_better,d_fold1,d_fold2,d_fold3,d_fold4,d_fold5,keep,weights_lgb_xgb_cb
variant,,,,,,,,,,
PM,76.970,-0.110,5,-0.162,-0.124,-0.074,-0.141,-0.048,True,"[0.256, 0.222, 0.522]"
H2,76.993,-0.086,5,-0.115,-0.083,-0.192,-0.041,-0.000,True,"[0.262, 0.25, 0.489]"
E,77.007,-0.072,4,-0.120,-0.078,-0.156,-0.040,0.032,True,"[0.275, 0.25, 0.475]"
R2,77.079,0.000,0,0.000,0.000,0.000,0.000,0.000,False,"[0.265, 0.251, 0.484]"


达标的新块： ['H2', 'E', 'PM'] → ALL3 = R2 + ['H2', 'E', 'PM']


## 9. ALL3：R2 + 全部达标的块（五折）→ 双胞胎后处理 → 导出提交文件

In [ ]:
if KEPT:
    for model in MODELS:
        if runs(model, 'ALL3'):
            Xtr, Xte, cat_cols = frames3('ALL3', model)
            p_val, p_test, _ = fit_predict(model, Xtr.iloc[pa], y[pa], Xtr.iloc[ps], y[ps], Xtr.iloc[pv],
                                           Xte.iloc[:500], cat_cols, rounds=30)          # ALL3 的预检
            require(np.isfinite(p_val).all() and np.isfinite(p_test).all(), f'ALL3 {model} 预检失败')
    run_variants(['ALL3'])
    BLENDS['ALL3'] = blend_variant('ALL3')
    final_table = paired_table({v: BLENDS[v]['oof'] for v in ['R2'] + KEPT + ['ALL3']})
    display(final_table.round(3))
    final_table.to_csv(OUTPUT_DIR / 'summary_blend_all3.csv')
    FINAL = 'ALL3'
else:
    print('没有达标的新块，保持 R2。')
    FINAL = 'R2'

CORE = CORE9


def row_keys(frame, columns):
    return pd.util.hash_pandas_object(frame[columns], index=False).to_numpy()


def mate_stats(train_keys, query_keys=None):
    table = pd.DataFrame({'k': train_keys, 'y': y}).groupby('k')['y'].agg(['sum', 'count'])
    if query_keys is None:                      # 训练行：留一
        keys = pd.Series(train_keys)
        n = keys.map(table['count']).to_numpy() - 1
        s = keys.map(table['sum']).to_numpy() - y
    else:                                       # 测试行：全部训练行
        keys = pd.Series(query_keys)
        n = keys.map(table['count']).fillna(0).to_numpy()
        s = keys.map(table['sum']).fillna(0).to_numpy()
    return n, np.where(n > 0, s / np.maximum(n, 1), np.nan)


def twins(query_exact=None, query_core=None):
    n_ex, m_ex = mate_stats(row_keys(train, RAW_FEATURES), query_exact)
    n_co, m_co = mate_stats(row_keys(train, CORE), query_core)
    level = np.select([n_ex > 0, n_co > 0], ['exact', 'core'], default='none')
    return level, np.where(level == 'exact', m_ex, m_co)


def fit_alpha(mask, pred, mate):
    d = mate[mask] - pred[mask]
    denom = float(np.sum(d * d))
    return float(np.clip(np.sum((y[mask] - pred[mask]) * d) / denom, 0, 1)) if denom > 0 else 0.0


def apply_twins(pred, mate, level, alphas):
    out = pred.copy()
    for lv, a in alphas.items():
        m = level == lv
        out[m] = pred[m] + a * (mate[m] - pred[m])
    return out


LEVEL, MATE = twins()
T_LEVEL, T_MATE = twins(row_keys(test, RAW_FEATURES), row_keys(test, CORE))
HALF = (pd.util.hash_pandas_object(train[['Sex', 'Country', 'State']], index=False).to_numpy() % 2).astype(int)
twin_rows = []
for name in (['R2', FINAL] if FINAL != 'R2' else ['R2']):
    oof, test_pred = BLENDS[name]['oof'], BLENDS[name]['test']
    alphas = {lv: fit_alpha(LEVEL == lv, oof, MATE) for lv in ['exact', 'core']}
    crossfit = oof.copy()
    for h in (0, 1):
        a = {lv: fit_alpha((HALF != h) & (LEVEL == lv), oof, MATE) for lv in ['exact', 'core']}
        crossfit[HALF == h] = apply_twins(oof[HALF == h], MATE[HALF == h], LEVEL[HALF == h], a)
    fixed = apply_twins(test_pred, T_MATE, T_LEVEL, alphas)
    twin_rows.append({'name': name, 'oof_rmse': rmse(y, oof), 'oof_rmse_twinfix_crossfit': rmse(y, crossfit),
                      'alpha_exact': alphas['exact'], 'alpha_core': alphas['core']})
    for suffix, pred in [('', test_pred), ('_twinfix', fixed)]:
        path = OUTPUT_DIR / f'submission_{name}_blend3{suffix}.csv'
        pd.DataFrame({'id': test['id'], TARGET: np.clip(pred, 0, None)}).to_csv(path, index=False)
        print('写出', path.name)
    np.save(OUTPUT_DIR / f'oof_{name}_blend3.npy', oof)
    np.save(OUTPUT_DIR / f'test_{name}_blend3.npy', test_pred)
twin_table = pd.DataFrame(twin_rows).set_index('name')
display(twin_table.round(3))
twin_table.to_csv(OUTPUT_DIR / 'summary_twins.csv')

fold 1  lgb      ALL3  RMSE 77.787  最佳轮数   663  (26s)
fold 1  xgb      ALL3  RMSE 78.328  最佳轮数   594  (12s)
fold 1  catboost ALL3  RMSE 77.528  最佳轮数  2432  (99s)
—— fold 1 完成，本次运行用时 2.3 分钟
fold 2  lgb      ALL3  RMSE 77.901  最佳轮数   296  (14s)
fold 2  xgb      ALL3  RMSE 78.907  最佳轮数   431  (10s)
fold 2  catboost ALL3  RMSE 77.474  最佳轮数  2734  (110s)
—— fold 2 完成，本次运行用时 2.2 分钟
fold 3  lgb      ALL3  RMSE 78.088  最佳轮数   388  (18s)
fold 3  xgb      ALL3  RMSE 78.747  最佳轮数   576  (12s)
fold 3  catboost ALL3  RMSE 77.889  最佳轮数  3226  (129s)
—— fold 3 完成，本次运行用时 2.7 分钟
fold 4  lgb      ALL3  RMSE 77.958  最佳轮数   331  (16s)
fold 4  xgb      ALL3  RMSE 78.549  最佳轮数   549  (12s)
fold 4  catboost ALL3  RMSE 77.674  最佳轮数  1946  (81s)
—— fold 4 完成，本次运行用时 1.8 分钟
fold 5  lgb      ALL3  RMSE 76.805  最佳轮数   547  (22s)
fold 5  xgb      ALL3  RMSE 77.753  最佳轮数   517  (11s)
fold 5  catboost ALL3  RMSE 76.960  最佳轮数  3117  (125s)
—— fold 5 完成，本次运行用时 2.6 分钟


,oof_rmse,mean_delta,folds_better,d_fold1,d_fold2,d_fold3,d_fold4,d_fold5,keep
variant,,,,,,,,,
ALL3,76.876,-0.203,5,-0.187,-0.195,-0.268,-0.190,-0.176,True
PM,76.970,-0.110,5,-0.162,-0.124,-0.074,-0.141,-0.048,True
H2,76.993,-0.086,5,-0.115,-0.083,-0.192,-0.041,-0.000,True
E,77.007,-0.072,4,-0.120,-0.078,-0.156,-0.040,0.032,True
R2,77.079,0.000,0,0.000,0.000,0.000,0.000,0.000,False


写出 submission_R2_blend3.csv
写出 submission_R2_blend3_twinfix.csv
写出 submission_ALL3_blend3.csv
写出 submission_ALL3_blend3_twinfix.csv


,oof_rmse,oof_rmse_twinfix_crossfit,alpha_exact,alpha_core
name,,,,
R2,77.079,76.846,0.859,0.523
ALL3,76.876,76.649,0.856,0.511


## 10. 第二层平滑实验（CPU，几分钟）

在 `FINAL` 融合加双胞胎之后，比较 5 种「同一运动员」残差平滑：

|做法|特征|
|---|---|
|`S0_exact`|08 原版：键 A0（性别 + 国家 + 州 + 出生年）、A1（+ 联盟）、A2（+ 装备）、A3（+ 级别），以及比赛键 M2；同键其他训练行的残差和 ÷（个数 + 收缩常数）|
|`S1_fuzzy`|同上，但出生年允许 ±1（整数年龄推算的出生年可能差 1）|
|`S2_decay`|同上，按比赛年份差加权：权重 exp(−年份差 / 3)，最多 ±8 年|
|`S3_all_linear`|三种特征合在一起，线性组合|
|`S4_all_gbm`|三种特征合在一起，小 LightGBM|

全部按「性别 + 国家 + 州」的哈希奇偶分两半交叉拟合（同一运动员的所有记录在同一半）：一半拟合、另一半评分。

In [ ]:
base_oof, base_test = BLENDS[FINAL]['oof'], BLENDS[FINAL]['test']
TR, TE = train.reset_index(drop=True), test.reset_index(drop=True)


def text_key(frame, columns):
    out = frame[columns[0]].astype(str)
    for c in columns[1:]:
        out = out + '|' + frame[c].astype(str)
    return out


BY_TR = np.floor(TR['MeetYear'] - TR['Age'] + 1e-9).astype(int)
BY_TE = np.floor(TE['MeetYear'] - TE['Age'] + 1e-9).astype(int)
YR_TR, YR_TE = TR['MeetYear'].astype(int), TE['MeetYear'].astype(int)
PREFIX = {'A0': ['Sex', 'Country', 'State'], 'A1': ['Sex', 'Country', 'State', 'Federation'],
          'A2': ['Sex', 'Country', 'State', 'Federation', 'Equipment'],
          'A3': ['Sex', 'Country', 'State', 'Federation', 'Equipment', 'WeightClassKg']}
SHRINK = {'A0': 5.0, 'A1': 5.0, 'A2': 3.0, 'A3': 2.0}
PFX_TR = {k: text_key(TR, c) for k, c in PREFIX.items()}
PFX_TE = {k: text_key(TE, c) for k, c in PREFIX.items()}
MEET_COLS = ['Federation', 'MeetYear', 'MeetCountry', 'MeetState', 'Sex', 'Equipment']
MEET_TR, MEET_TE = text_key(TR, MEET_COLS), text_key(TE, MEET_COLS)
VALID_TR = (TR['Country'] != 'Unknown').to_numpy(dtype='float64')
VALID_TE = (TE['Country'] != 'Unknown').to_numpy(dtype='float64')


def sums(keys, values):
    table = pd.DataFrame({'k': keys.to_numpy(), 'v': values}).groupby('k')['v'].agg(['sum', 'count'])
    return table['sum'], table['count']


def look(table, keys):
    s, n = table
    return keys.map(s).fillna(0).to_numpy(dtype='float64'), keys.map(n).fillna(0).to_numpy(dtype='float64')


def f_exact(k, r):
    ktr, kte = PFX_TR[k] + '|' + BY_TR.astype(str), PFX_TE[k] + '|' + BY_TE.astype(str)
    table = sums(ktr, r)
    s, n = look(table, ktr)
    st, nt = look(table, kte)
    return (s - r) / (n - 1 + SHRINK[k]), st / (nt + SHRINK[k])


def f_fuzzy(k, r):
    table = sums(PFX_TR[k] + '|' + BY_TR.astype(str), r)
    s, n, st, nt = np.zeros(n_train), np.zeros(n_train), np.zeros(len(TE)), np.zeros(len(TE))
    for d in (-1, 0, 1):
        a, b = look(table, PFX_TR[k] + '|' + (BY_TR + d).astype(str))
        s, n = s + a, n + b
        a, b = look(table, PFX_TE[k] + '|' + (BY_TE + d).astype(str))
        st, nt = st + a, nt + b
    return (s - r) / (n - 1 + SHRINK[k]), st / (nt + SHRINK[k])


def f_decay(k, r, tau=3.0, max_gap=8):
    base_tr = PFX_TR[k] + '|' + BY_TR.astype(str) + '|'
    base_te = PFX_TE[k] + '|' + BY_TE.astype(str) + '|'
    table = sums(base_tr + YR_TR.astype(str), r)
    s, n, st, nt = np.zeros(n_train), np.zeros(n_train), np.zeros(len(TE)), np.zeros(len(TE))
    for d in range(-max_gap, max_gap + 1):
        w = float(np.exp(-abs(d) / tau))
        a, b = look(table, base_tr + (YR_TR + d).astype(str))
        s, n = s + w * a, n + w * b
        a, b = look(table, base_te + (YR_TE + d).astype(str))
        st, nt = st + w * a, nt + w * b
    return (s - r) / (n - 1 + SHRINK[k]), st / (nt + SHRINK[k])     # d = 0 时自己权重为 1，已扣除


def f_meet(r, lam=20.0):
    table = sums(MEET_TR, r)
    s, n = look(table, MEET_TR)
    st, nt = look(table, MEET_TE)
    return (s - r) / (n - 1 + lam), st / (nt + lam)


def stage2_features(r, kind):
    builders = {'exact': f_exact, 'fuzzy': f_fuzzy, 'decay': f_decay}
    cols_tr, cols_te, names = [], [], []
    for kd in (['exact', 'fuzzy', 'decay'] if kind == 'all' else [kind]):
        for k in PREFIX:
            a, b = builders[kd](k, r)
            cols_tr.append(a * VALID_TR)
            cols_te.append(b * VALID_TE)
            names.append(f'{kd}_{k}')
    a, b = f_meet(r)
    cols_tr.append(a)
    cols_te.append(b)
    names.append('meet')
    return np.column_stack(cols_tr), np.column_stack(cols_te), names


alphas = {lv: fit_alpha(LEVEL == lv, base_oof, MATE) for lv in ['exact', 'core']}
p1 = apply_twins(base_oof, MATE, LEVEL, alphas)
p1_cf = base_oof.copy()
for h in (0, 1):
    a_h = {lv: fit_alpha((HALF != h) & (LEVEL == lv), base_oof, MATE) for lv in ['exact', 'core']}
    m = HALF == h
    p1_cf[m] = apply_twins(base_oof[m], MATE[m], LEVEL[m], a_h)
t1 = apply_twins(base_test, T_MATE, T_LEVEL, alphas)
r1 = y - p1
FEATS = {kind: stage2_features(r1, kind) for kind in ['exact', 'fuzzy', 'decay', 'all']}
GBM_PARAMS = dict(n_estimators=300, learning_rate=0.03, num_leaves=15, min_child_samples=300, subsample=0.8,
                  subsample_freq=1, reg_lambda=10.0, random_state=SEED, n_jobs=-1, verbose=-1)


def linear_fit_predict(F_fit, r_fit, F_apply):
    beta = np.linalg.lstsq(F_fit, r_fit, rcond=None)[0]
    return F_apply @ beta


def gbm_fit_predict(F_fit, r_fit, F_apply):
    return lgb.LGBMRegressor(**GBM_PARAMS).fit(F_fit, r_fit).predict(F_apply)


S2_VARIANTS = {'S0_exact': ('exact', linear_fit_predict), 'S1_fuzzy': ('fuzzy', linear_fit_predict),
               'S2_decay': ('decay', linear_fit_predict), 'S3_all_linear': ('all', linear_fit_predict),
               'S4_all_gbm': ('all', gbm_fit_predict)}
rows, S2_OOF, S2_TEST = [], {}, {}
for name, (kind, fn) in S2_VARIANTS.items():
    F, FT, names = FEATS[kind]
    pred_cf = p1_cf.copy()
    for h in (0, 1):
        fit, m = HALF != h, HALF == h
        pred_cf[m] = p1_cf[m] + fn(F[fit], r1[fit], F[m])
    S2_OOF[name], S2_TEST[name] = pred_cf, t1 + fn(F, r1, FT)
    rows.append({'variant': name, 'n_features': len(names), 'crossfit_rmse': rmse(y, pred_cf),
                 'gain_vs_twins': rmse(y, p1_cf) - rmse(y, pred_cf),
                 **{f'fold{k}': v for k, v in fold_rmse(pred_cf).items()}})
s2_table = pd.DataFrame(rows).set_index('variant').sort_values('crossfit_rmse')
print(f'{FINAL} 融合：OOF {rmse(y, base_oof):.3f} → 加双胞胎（交叉拟合）{rmse(y, p1_cf):.3f}')
display(s2_table.round(3))
s2_table.to_csv(OUTPUT_DIR / 'summary_stage2_variants.csv')
BEST_S2 = s2_table.index[0]
for name in dict.fromkeys(['S0_exact', BEST_S2]):
    path = OUTPUT_DIR / f'submission_{FINAL}_blend3_twins_{name}.csv'
    pd.DataFrame({'id': test['id'], TARGET: np.clip(S2_TEST[name], 0, None)}).to_csv(path, index=False)
    print('写出', path.name)
print(f'交叉拟合最好的第二层：{BEST_S2}')

ALL3 融合：OOF 76.876 → 加双胞胎（交叉拟合）76.649


,n_features,crossfit_rmse,gain_vs_twins,fold1,fold2,fold3,fold4,fold5
variant,,,,,,,,
S3_all_linear,13,76.510,0.139,76.489,76.677,76.906,76.740,75.733
S2_decay,5,76.527,0.122,76.516,76.688,76.893,76.764,75.770
S0_exact,5,76.539,0.109,76.503,76.693,76.938,76.766,75.792
S1_fuzzy,5,76.544,0.105,76.514,76.707,76.947,76.749,75.799
S4_all_gbm,13,76.858,-0.209,76.890,76.909,77.313,77.104,76.067


写出 submission_ALL3_blend3_twins_S0_exact.csv
写出 submission_ALL3_blend3_twins_S3_all_linear.csv
交叉拟合最好的第二层：S3_all_linear


## 11. 残差诊断（为最终集成找方向）

In [ ]:
def bucket(values, edges, labels):
    return pd.cut(values, edges, labels=labels, right=False).astype(str)


groups = {
    'age_half': P['age_half'].iloc[:n_train].to_numpy(),
    'ath_n': bucket(H['ath_n'].iloc[:n_train].fillna(-1), [-1, 0, 2, 3, 6, 1e9], ['未知', '1', '2', '3-5', '6+']),
    'ath_years_since_first': bucket(H['ath_years_since_first'].iloc[:n_train].fillna(-1), [-1, 0, 1, 2, 5, 100],
                                    ['未知', '0', '1', '2-4', '5+']),
    'ath_ever_abroad': H2['ath_ever_abroad'].iloc[:n_train].fillna(-1).to_numpy(),
    'ath_max_meet_states': bucket(H2['ath_max_meet_states'].iloc[:n_train].fillna(-1), [-1, 0, 1, 2, 5, 10, 1e9],
                                  ['未知', '0', '1', '2-4', '5-9', '10+']),
    'meet_n_states': bucket(M2['meet_n_states'].iloc[:n_train], [0, 1, 2, 5, 10, 1e9], ['0', '1', '2-4', '5-9', '10+']),
    'dup_exact_n>0': (D['dup_exact_n'].iloc[:n_train] > 0).to_numpy(),
    'Sex×Equipment': (train['Sex'] + '|' + train['Equipment']).to_numpy(),
    'Tested': train['Tested'].to_numpy(),
    'MeetYear 段': bucket(train['MeetYear'], [1900, 2000, 2010, 2015, 2020, 2030],
                          ['<2000', '2000s', '2010-14', '2015-19', '2020+']),
}
columns = {'R2': BLENDS['R2']['oof'], FINAL: BLENDS[FINAL]['oof'], f'{FINAL}+双胞胎+{BEST_S2}': S2_OOF[BEST_S2]}
frames = []
for label, pred in columns.items():
    res = y - pred
    parts = []
    for g, key in groups.items():
        part = pd.DataFrame({'k': key, 'r': res}).groupby('k')['r'].agg(['size', 'mean'])
        part.index = pd.MultiIndex.from_product([[g], part.index.astype(str)], names=['signal', 'level'])
        parts.append(part)
    frames.append(pd.concat(parts).rename(columns={'mean': label}))
diag = pd.concat(frames, axis=1)
diag = diag.loc[:, ~diag.columns.duplicated()]
display(diag.round(2))
diag.to_csv(OUTPUT_DIR / 'diagnostics_residuals.csv')
print('完成。跑完告诉 Claude，本地会从 Drive 同步 OUTPUT_DIR 的结果。')

size      R2   ALL3  \
signal                level                                 
age_half              0.0            54422    0.15   0.22   
                      1.0            50578   -0.86  -0.85   
ath_n                 1              14513   -1.70  -1.62   
                      2               9608   -0.82  -0.94   
                      3-5            16275    0.21   0.23   
                      6+             62598   -0.10  -0.03   
                      未知              2006    0.09  -0.02   
ath_years_since_first 0              32018   -1.25  -1.25   
                      1               8688    0.48   0.77   
                      2-4            24185    0.14   0.27   
                      5+             38103   -0.08  -0.10   
                      未知              2006    0.09  -0.02   
ath_ever_abroad       -1.0            2006    0.09  -0.02   
                      0.0            90551   -0.53  -0.35   
                      1.0            12443    1.01   0.11   
ath_max_meet_states   0              53639   -0.39  -0.39   
                      1               5304   -1.67  -1.31   
                      10+            25649    0.26  -0.01   
                      2-4             8970   -0.83  -0.19   
                      5-9             9432   -0.54  -0.05   
                      未知              2006    0.09  -0.02   
meet_n_states         0              63228   -0.35  -0.34   
                      1               5197   -1.73  -1.38   
                      10+            11636    0.61   0.35   
                      2-4            12016   -1.02  -0.64   
                      5-9            12923    0.05   0.13   
dup_exact_n>0         False         103432   -0.46  -0.40   
                      True            1568    8.04   7.03   
Sex×Equipment         F|Multi-ply      380    6.71   7.16   
                      F|Raw          21755   -1.31  -1.10   
                      F|Single-ply    6366   -1.62  -1.12   
                      F|Unlimited      521    1.57   2.08   
                      F|Wraps         3515   -0.92   0.03   
                      Mx|Raw             8  -27.99 -35.12   
                      Mx|Wraps           1  120.11  90.30   
                      M|Multi-ply     2053    1.67   1.58   
                      M|Raw          43471   -0.26  -0.27   
                      M|Single-ply   17352    0.57   0.43   
                      M|Unlimited      545    2.49   2.60   
                      M|Wraps         9033    0.01  -0.40   
Tested                Untested       24982    0.43   0.57   
                      Yes            80018   -0.58  -0.56   
MeetYear 段            2000s           7980   -0.32  -0.58   
                      2010-14        10672   -0.73  -0.62   
                      2015-19        30764   -0.51  -0.45   
                      2020+          51112   -0.14  -0.09   
                      <2000           4472   -0.49  -0.22   

                                    ALL3+双胞胎+S3_all_linear  
signal                level                                 
age_half              0.0                             0.13  
                      1.0                            -0.94  
ath_n                 1                              -1.58  
                      2                              -1.15  
                      3-5                             0.03  
                      6+                             -0.11  
                      未知                             -0.18  
ath_years_since_first 0                              -1.36  
                      1                               0.48  
                      2-4                             0.18  
                      5+                             -0.14  
                      未知                             -0.18  
ath_ever_abroad       -1.0                           -0.18  
                      0.0                            -0.43  
                      1.0                            -0.09  
ath_max_mee

完成。跑完告诉 Claude，本地会从 Drive 同步 OUTPUT_DIR 的结果。
